In [15]:
import os
from dotenv import load_dotenv

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")


In [16]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, refresh_schema=False
)
graph

d:\GraphLangchain\venv\Lib\site-packages\langchain_community\graphs\neo4j_graph.py:205: ExperimentalWarning: The configuration may change in the future.
  self._driver.verify_connectivity()


In [17]:
## Dataset Moview 
movie_query="""

LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""

In [18]:
from langchain_groq import ChatGroq
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model = "llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001882CA6F320>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001882CA6F830>, model_name='llama-3.1-8b-instant', groq_api_key=SecretStr('**********'))

In [19]:
graph.query(movie_query)

[]

In [20]:
# from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
from langchain.chains import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True   # IMPORTANT (newer versions)
)

chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x000001881235CE90>, cypher_generation_chain=LLMChain(prompt=PromptTemplate(input_variables=['question', 'schema'], template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001882CA6F320>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001882CA6F830>, model_name='llama-3.1-8b-instant', groq_api_key=SecretStr('*****

In [21]:
chain.schema

<bound method BaseModel.schema of <class 'langchain.chains.graph_qa.cypher.GraphCypherQAChain'>>

In [22]:
chain.graph_schema

'Node properties are the following:\n\nRelationship properties are the following:\n\nThe relationships are the following:\n'

In [ ]:
# https://github.com/tomasonjo/blog-datasets/blob/main/movies/movies.csv
examples = [
    {
        "question": "How many artists are there?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)",
    },
    {
        "question": "Which actors played in the movie Casino?",
        "query": "MATCH (m:Movie)-[:ACTED_IN]-(a:Person) WHERE m.title CONTAINS 'Casino' RETURN a.name",
    },
    {
        "question": "How many movies has Tom Hanks acted in?",
        "query": "MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)",
    },
    {
        "question": "List all the genres of the movie Schindler's List",
        "query": "MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WHERE m.title CONTAINS 'Schindler\\'s List' RETURN g.name",
    },
    {
        "question": "Which actors have worked in both comedy and action movies?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(m1:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(m2:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name",
    },
    {
        "question": "Which directors have made movies with at least three actors named John?",
        "query": "MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person) WHERE a.name STARTS WITH 'John' WITH d, COUNT(DISTINCT a) AS JohnsCount WHERE JohnsCount >= 3 RETURN d.name",
    },
    {
        "question": "Find movies where the director also acted in the movie",
        "query": "MATCH (p:Person)-[:DIRECTED]->(m:Movie), (p)-[:ACTED_IN]->(m) RETURN m.title, p.name",
    },
    {
        "question": "Find the actor who has acted in the most movies",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) RETURN a.name, COUNT(m) AS movieCount ORDER BY movieCount DESC LIMIT 1",
    },
    {
        "question": "Who directed the movie Inception?",
        "query": "MATCH (m:Movie)<-[:DIRECTED]-(d:Person) WHERE m.title CONTAINS 'Inception' RETURN d.name",
    },
    {
        "question": "List all movies directed by Christopher Nolan",
        "query": "MATCH (d:Person {name: 'Christopher Nolan'})-[:DIRECTED]->(m:Movie) RETURN m.title",
    },
    {
        "question": "Find all movies where Leonardo DiCaprio acted",
        "query": "MATCH (a:Person {name: 'Leonardo DiCaprio'})-[:ACTED_IN]->(m:Movie) RETURN m.title",
    },
    {
        "question": "How many movies are in the database?",
        "query": "MATCH (m:Movie) RETURN count(m)",
    },
    {
        "question": "What are the top 5 highest rated movies?",
        "query": "MATCH (m:Movie) RETURN m.title, m.imdbRating ORDER BY m.imdbRating DESC LIMIT 5",
    },
    {
        "question": "Find movies released after 2010",
        "query": "MATCH (m:Movie) WHERE m.released > 2010 RETURN m.title, m.released",
    },
    {
        "question": "Which genres does Tom Hanks usually act in?",
        "query": "MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)-[:IN_GENRE]->(g:Genre) RETURN DISTINCT g.name",
    },
    {
        "question": "Find actors who have worked with Brad Pitt",
        "query": "MATCH (a:Person {name: 'Brad Pitt'})-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(coActor:Person) RETURN DISTINCT coActor.name",
    },
    {
        "question": "List all genres available in the database",
        "query": "MATCH (g:Genre) RETURN DISTINCT g.name",
    },
    {
        "question": "Find movies that belong to both Drama and Romance genres",
        "query": "MATCH (m:Movie)-[:IN_GENRE]->(g1:Genre), (m)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Drama' AND g2.name = 'Romance' RETURN DISTINCT m.title",
    }
]

In [35]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template(
    "User input: {question}\n Cypher query:{query}"
)

prompt = FewShotPromptTemplate(
    examples=examples[:5],
    example_prompt=example_prompt,
    prefix="You are a Neo4j expert. Given an input question, create a syntatically very accurate Cypher Query.",
    suffix="User input:{question}\nCypher query",
    input_variables=["question", "schema"]
)

In [36]:
prompt

FewShotPromptTemplate(input_variables=['question'], examples=[{'question': 'How many artists are there?', 'query': 'MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)'}, {'question': 'Which actors played in the movie Casino?', 'query': "MATCH (m:Movie)-[:ACTED_IN]-(a:Person) WHERE m.title CONTAINS 'Casino' RETURN a.name"}, {'question': 'How many movies has Tom Hanks acted in?', 'query': "MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)"}, {'question': "List all the genres of the movie Schindler's List", 'query': "MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WHERE m.title CONTAINS 'Schindler\\'s List' RETURN g.name"}, {'question': 'Which actors have worked in both comedy and action movies?', 'query': "MATCH (a:Person)-[:ACTED_IN]->(m1:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(m2:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name"}], example_prompt=PromptTemplate(input_variables=['query', 'ques

In [37]:
schema = graph.schema
print(prompt.format(question="How many artists are there?", schema="foo"))

You are a Neo4j expert. Given an input question, create a syntatically very accurate Cypher Query.

User input: How many artists are there?
 Cypher query:MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)

User input: Which actors played in the movie Casino?
 Cypher query:MATCH (m:Movie)-[:ACTED_IN]-(a:Person) WHERE m.title CONTAINS 'Casino' RETURN a.name

User input: How many movies has Tom Hanks acted in?
 Cypher query:MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)

User input: List all the genres of the movie Schindler's List
 Cypher query:MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WHERE m.title CONTAINS 'Schindler\'s List' RETURN g.name

User input: Which actors have worked in both comedy and action movies?
 Cypher query:MATCH (a:Person)-[:ACTED_IN]->(m1:Movie)-[:IN_GENRE]->(g1:Genre), (a)-[:ACTED_IN]->(m2:Movie)-[:IN_GENRE]->(g2:Genre) WHERE g1.name = 'Comedy' AND g2.name = 'Action' RETURN DISTINCT a.name

User input:How many artists are there

In [38]:
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001882CA6F320>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001882CA6F830>, model_name='llama-3.1-8b-instant', groq_api_key=SecretStr('**********'))

In [40]:
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, cypher_prompt=prompt, verbose=True)

In [42]:
chain.invoke("Which actors played in movie casino ?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (m:Movie)-[:ACTED_IN]-(a:Person) WHERE m.title CONTAINS 'Casino' RETURN a.name

Full Context:
[{'a.name': 'Robert De Niro'}, {'a.name': 'Joe Pesci'}, {'a.name': 'Sharon Stone'}, {'a.name': 'James Woods'}]

> Finished chain.


{'query': 'Which actors played in movie casino ?',
 'result': 'Robert De Niro, Joe Pesci, Sharon Stone, and James Woods played in the movie Casino.'}

In [46]:
chain.invoke("How many movies has Tom Hanks acted in?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
sql
MATCH (a:Person) RETURN count(DISTINCT a)



ValueError: Generated Cypher Statement is not valid
{code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'sql': expected 'ALTER', 'ORDER BY', 'CALL', 'CREATE', 'LOAD CSV', 'START DATABASE', 'STOP DATABASE', 'DEALLOCATE', 'DELETE', 'DENY', 'DETACH', 'DROP', 'DRYRUN', 'FILTER', 'FINISH', 'FOREACH', 'GRANT', 'INSERT', 'LET', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REALLOCATE', 'REMOVE', 'RENAME', 'RETURN', 'REVOKE', 'ENABLE SERVER', 'SET', 'SHOW', 'SKIP', 'TERMINATE', 'UNWIND', 'USE', 'WHEN', 'WITH' or '{' (line 1, column 1 (offset: 0))
"sql"
 ^}

In [47]:
chain.invoke("Which genres does Tom Hanks usually act in?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (a:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)-[:IN_GENRE]->(g:Genre)
RETURN g.name AS genre, COUNT(g) AS frequency
ORDER BY frequency DESC
LIMIT 10

Full Context:
[{'genre': 'Adventure', 'frequency': 2}, {'genre': 'Animation', 'frequency': 1}, {'genre': 'Children', 'frequency': 1}, {'genre': 'Comedy', 'frequency': 1}, {'genre': 'Fantasy', 'frequency': 1}, {'genre': 'Drama', 'frequency': 1}, {'genre': 'IMAX', 'frequency': 1}]

> Finished chain.


{'query': 'Which genres does Tom Hanks usually act in?',
 'result': "I don't know the answer."}